# Day 20: Same Problem, 3 Frameworks

**30-Day AI Agent Challenge: OpenAI Agents SDK vs LangGraph vs CrewAI**

---

## Today's Concept

> The ultimate test: build the SAME agent system in all 3 frameworks.

We've learned each framework separately. Now we tackle one real problem —
a customer support triage system — and build it three ways. This reveals
the true tradeoffs: ceremony, readability, flexibility, and effort.

## What You'll Build
- **Customer Support Triage**: Classifies incoming questions, routes to the right specialist,
  gets an answer, and formats a professional response
- The SAME logic in all 3 frameworks
- An honest comparison based on actual lines of code and developer experience

In [6]:
# ── Setup ───────────────────────────────────────────────
import sys
sys.path.insert(0, "..")
from shared import check_and_report, print_config, disable_openai_tracing, get_openai_agents_model, get_langgraph_model, get_crewai_llm
check_and_report()
disable_openai_tracing()
print("=" * 50)
print("DAY 20 SETUP COMPLETE")
print("=" * 50)
print_config()

Python: 3.12.13

All required packages installed.
Optional (missing): langchain-openai, langgraph-checkpoint-memory, ipywidgets

Ollama: connected (10 model(s) available)

DAY 20 SETUP COMPLETE
  Provider: OLLAMA
  Model:    qwen2.5:3b
  Ollama:   http://localhost:11434


## Today's Problem

> **Build a customer support triage agent that:**
> 1. Classifies incoming questions (billing, technical, general)
> 2. Routes to the right specialist agent
> 3. Gets an answer from the specialist
> 4. Formats a professional response

**Test queries:**
- "I can't log into my account, it keeps saying invalid password"
- "I was charged twice for my last order #12345"
- "What are your business hours?"

This is a realistic multi-agent pattern used in production systems.

---
## Framework 1: OpenAI Agents SDK

The OpenAI SDK handles routing via `handoffs` — the LLM decides which agent
to transfer to. This is the most "AI-native" approach.

In [11]:
from agents import Agent, Runner, function_tool
import time

model = get_openai_agents_model()

# Knowledge base for specialists
billing_kb = {
    "refund": "Refunds take 3-5 business days to process. You can check status in your account under Orders.",
    "charge": "Duplicate charges are investigated within 24 hours. Please provide your order number.",
    "invoice": "Invoices are available under Account > Billing > Invoice History.",
}
tech_kb = {
    "login": "Try resetting your password at /reset. If that fails, clear your browser cache and cookies.",
    "slow": "Check your internet connection. If the issue persists, try disabling browser extensions.",
    "error": "Note the error code and timestamp. Our team can investigate with those details.",
}
general_kb = {
    "hours": "We're available Monday-Friday, 9 AM - 6 PM EST.",
    "contact": "Email us at support@example.com or call 1-800-555-0123.",
    "location": "Our office is at 123 Tech Street, San Francisco, CA.",
}

test_queries = [
    "I can't log into my account, it keeps saying invalid password",
    "I was charged twice for my last order #12345",
    "What are your business hours?",
]

# ── OpenAI Agents SDK Implementation ──
# Agent names use underscores so the auto-generated handoff tool names
# (transfer_to_<agent_name>) are clean and produce no transformation warnings.
# handoff_description on each specialist is picked up automatically when the
# triage agent lists them in handoffs=[...] — no need to call handoff().

@function_tool
def lookup_billing(query: str) -> str:
    """Look up billing information."""
    for key, val in billing_kb.items():
        if key in query.lower():
            return val
    return "No specific billing info found. Escalating to a human agent."

@function_tool
def lookup_tech(query: str) -> str:
    """Look up technical support information."""
    for key, val in tech_kb.items():
        if key in query.lower():
            return val
    return "No fix found. Escalating to the engineering team."

@function_tool
def lookup_general(query: str) -> str:
    """Look up general company information."""
    for key, val in general_kb.items():
        if key in query.lower():
            return val
    return "No general info found. Connecting you with a team member."

# Specialist agents. handoff_description tells the triage model when to pick each.
billing_specialist = Agent(
    name="billing_specialist",
    handoff_description="For billing questions: charges, refunds, invoices, payments.",
    instructions=(
        "You handle billing questions. Call the lookup_billing tool to find answers. "
        "Be empathetic and professional. Format responses clearly."
    ),
    tools=[lookup_billing],
    model=model,
)

tech_support = Agent(
    name="tech_support",
    handoff_description="For technical issues: login, errors, performance, bugs.",
    instructions=(
        "You handle technical issues. Call the lookup_tech tool to find solutions. "
        "Give step-by-step instructions. Be patient and clear."
    ),
    tools=[lookup_tech],
    model=model,
)

general_info = Agent(
    name="general_info",
    handoff_description="For general questions: hours, contact info, location.",
    instructions=(
        "You handle general inquiries. Call the lookup_general tool. "
        "Be friendly and concise."
    ),
    tools=[lookup_general],
    model=model,
)

# Triage agent. The default tool names are transfer_to_<agent_name>; we list
# them explicitly so a small model knows exactly which to call.
triage = Agent(
    name="triage",
    instructions=(
        "You are a customer support triage agent. "
        "Your ONLY job is to call exactly ONE of these tools, then stop:\n"
        "- transfer_to_billing_specialist: charges, refunds, invoices, payments\n"
        "- transfer_to_tech_support: login, errors, performance, bugs\n"
        "- transfer_to_general_info: hours, contact info, location\n\n"
        "Rules:\n"
        "1. Read the customer's question.\n"
        "2. Call exactly ONE transfer_to_* tool. Do not ask follow-up questions.\n"
        "3. Do NOT write a response yourself. The specialist handles that."
    ),
    handoffs=[billing_specialist, tech_support, general_info],
    model=model,
)

print("=" * 60)
print("OPENAI AGENTS SDK — CUSTOMER SUPPORT TRIAGE")
print("=" * 60)

# ponytail: substring heuristic — flags responses that read as escalation
# rather than a real answer. Catches the small-model failure mode where the
# handoff succeeds but the specialist doesn't call its lookup tool, so the
# response is a "we'll look into it" message instead of KB-grounded advice.
_ESCALATION_MARKERS = (
    "escalat", "will be in touch", "look into", "will investigate",
    "will review", "flagged this", "hold while we connect",
    "directed our", "we will be in contact", "while we connect",
)

openai_results = []
for q in test_queries:
    print(f"\n>>> CUSTOMER: {q}")
    start = time.time()
    result = await Runner.run(triage, q, max_turns=10)
    elapsed = time.time() - start
    out = (result.final_output or "").lower()
    if result.last_agent.name == "triage":
        print("  [WARN] triage did not hand off — model kept control")
    elif any(m in out for m in _ESCALATION_MARKERS):
        print(f"  [ROUTED TO] → {result.last_agent.name}  "
              "[WARN] response reads as escalation — specialist may not have used its KB tool")
    else:
        print(f"  [ROUTED TO] → {result.last_agent.name}")
    print(f"\nRESPONSE ({elapsed:.1f}s): {result.final_output}")
    openai_results.append(result.final_output)

OPENAI AGENTS SDK — CUSTOMER SUPPORT TRIAGE

>>> CUSTOMER: I can't log into my account, it keeps saying invalid password
  [ROUTED TO] → tech_support

RESPONSE (18.3s): Could you please provide more details? When attempting to log in, did the error message mention any specific account information that might be incorrect or missing? Also, how are you trying to enter your password? Are you using a web browser or from an app? Providing these specifics will help us investigate further.

>>> CUSTOMER: I was charged twice for my last order #12345
  [ROUTED TO] → billing_specialist  [WARN] response reads as escalation — specialist may not have used its KB tool

RESPONSE (9.9s): It seems there was an error in processing your order #12345. I have transferred this issue to our billing specialist for further investigation. They will be in touch with you shortly to resolve the double charge issue.
I'll keep trying to assist you directly, but if it takes longer than usual, your call might be forwar

---
## Framework 2: LangGraph

LangGraph uses explicit graph topology: a classification node determines the
route, then edges direct to specialist nodes. The most "engineered" approach.

In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict, Annotated
import operator

model = get_langgraph_model()

# ── LangGraph Implementation ──
# ponytail: KB is baked into each node's prompt rather than wrapped in @tool
# functions. The small local model answers more reliably when the KB is already
# in context than when it has to decide to call a tool first.

class SupportState(TypedDict):
    messages: Annotated[list, operator.add]
    query: str
    category: str
    response: str

def classify_node(state: SupportState) -> dict:
    """Classify the query into a category."""
    r = model.invoke([
        SystemMessage(content=(
            "Classify this customer query into EXACTLY one category: "
            "billing, technical, or general.\n"
            "Examples:\n"
            "  'I was charged twice'        → billing\n"
            "  'Refund my order'            → billing\n"
            "  'I can't log in'             → technical\n"
            "  'The app is slow'            → technical\n"
            "  'What are your hours?'       → general\n"
            "  'Where are you located?'     → general\n"
            "  'How do I contact you?'      → general\n"
            "Reply with ONLY the category name, nothing else."
        )),
        HumanMessage(content=state["query"]),
    ])
    raw = r.content.lower().strip()
    # ponytail: exact match first; substring fallback only if the model was verbose
    if raw.startswith("billing"):
        category = "billing"
    elif raw.startswith("tech"):
        category = "technical"
    elif raw.startswith("general"):
        category = "general"
    elif "billing" in raw:
        category = "billing"
    elif "tech" in raw:
        category = "technical"
    else:
        category = "general"
    print(f"  [CLASSIFY] raw={raw!r} → {category}")
    return {"category": category}

def billing_node(state: SupportState) -> dict:
    """Handle billing questions."""
    r = model.invoke([
        SystemMessage(content="You handle billing. Be empathetic and professional."),
        HumanMessage(content=f"Customer question: {state['query']}\n\nBilling info: {billing_kb}"),
    ])
    return {"response": r.content, "messages": [r]}

def technical_node(state: SupportState) -> dict:
    """Handle technical issues."""
    r = model.invoke([
        SystemMessage(content="You handle technical support. Give step-by-step instructions."),
        HumanMessage(content=f"Customer question: {state['query']}\n\nTech info: {tech_kb}"),
    ])
    return {"response": r.content, "messages": [r]}

def general_node(state: SupportState) -> dict:
    """Handle general inquiries."""
    r = model.invoke([
        SystemMessage(content="You handle general inquiries. Be friendly and concise."),
        HumanMessage(content=f"Customer question: {state['query']}\n\nGeneral info: {general_kb}"),
    ])
    return {"response": r.content, "messages": [r]}

def route_by_category(state: SupportState) -> str:
    """Route to the appropriate specialist."""
    return state.get("category", "general")

builder = StateGraph(SupportState)
builder.add_node("classify", classify_node)
builder.add_node("billing", billing_node)
builder.add_node("technical", technical_node)
builder.add_node("general", general_node)

builder.add_edge(START, "classify")
builder.add_conditional_edges(
    "classify",
    route_by_category,
    {"billing": "billing", "technical": "technical", "general": "general"},
)
builder.add_edge("billing", END)
builder.add_edge("technical", END)
builder.add_edge("general", END)

graph = builder.compile()

print("=" * 60)
print("LANGGRAPH — CUSTOMER SUPPORT TRIAGE")
print("=" * 60)

langgraph_results = []
for q in test_queries:
    print(f"\n>>> CUSTOMER: {q}")
    start = time.time()
    result = graph.invoke({"messages": [], "query": q, "category": "", "response": ""})
    elapsed = time.time() - start
    print(f"\nRESPONSE ({elapsed:.1f}s): {result['response'][:300]}")
    langgraph_results.append(result['response'][:200])

LANGGRAPH — CUSTOMER SUPPORT TRIAGE

>>> CUSTOMER: I can't log into my account, it keeps saying invalid password
  [CLASSIFY] raw='technical' → technical

RESPONSE (45.9s): Based on the information provided by the tech info, here are step-by-step instructions for troubleshooting the login issues:

### Step 1: Reset Your Password
1. **Access your account reset page:** Navigate to `/reset` in your web browser.
2. **Enter your email address:** Provide your registered emai

>>> CUSTOMER: I was charged twice for my last order #12345
  [CLASSIFY] raw='billing' → billing

RESPONSE (12.0s): Hello,

I appreciate you bringing this to our attention. We have noted the duplicate charge for Order #12345 and we are currently investigating it.

Could you please confirm if you've seen any recent charges in your account that were not part of your previous order? It would also be helpful if you c

>>> CUSTOMER: What are your business hours?
  [CLASSIFY] raw='general' → general

RESPONSE (6.7s): Our busin

---
## Framework 3: CrewAI

CrewAI uses roles and tasks — the most "organizational" approach. Each agent
has a clear role, and tasks flow sequentially.

In [13]:
from crewai import Agent, Task, Crew, Process

llm = get_crewai_llm()

# ── CrewAI Implementation ──
# CrewAI has no built-in router. We classify with a 1-task crew, pick the
# specialist in Python, then run a 2-task answer+format crew. This is the
# honest CrewAI pattern: roles + sequential tasks, manual routing.

triage_agent = Agent(
    role="Support Triage Specialist",
    goal="Classify customer questions accurately",
    backstory="You're the first point of contact for customer support.",
    llm=llm, verbose=False,
)
billing_specialist = Agent(
    role="Billing Specialist",
    goal="Resolve billing questions empathetically",
    backstory="Experienced billing support agent.",
    llm=llm, verbose=False,
)
tech_specialist = Agent(
    role="Technical Support Engineer",
    goal="Resolve technical issues with clear steps",
    backstory="Patient tech support engineer.",
    llm=llm, verbose=False,
)
general_specialist = Agent(
    role="Customer Service Representative",
    goal="Answer general questions concisely",
    backstory="Friendly front-desk agent.",
    llm=llm, verbose=False,
)
formatter_agent = Agent(
    role="Response Formatter",
    goal="Format specialist answers into professional responses",
    backstory="You polish specialist answers into customer-facing messages.",
    llm=llm, verbose=False,
)

specialists = {
    "billing": billing_specialist,
    "technical": tech_specialist,
    "general": general_specialist,
}
# ponytail: pass only the relevant KB to the specialist. Earlier version dumped
# all three KBs into every task; that confused the small model.
kbs = {"billing": billing_kb, "technical": tech_kb, "general": general_kb}

print("=" * 60)
print("CREWAI — CUSTOMER SUPPORT TRIAGE")
print("=" * 60)

crewai_results = []
for q in test_queries:
    print(f"\n>>> CUSTOMER: {q}")

    # Step 1: classify with a 1-task crew
    classify_task = Task(
        description=(
            f"Classify this customer question: '{q}'\n"
            f"Reply with EXACTLY ONE word: billing, technical, or general."
        ),
        expected_output="One word: billing, technical, or general",
        agent=triage_agent,
    )
    classify_crew = Crew(
        agents=[triage_agent],
        tasks=[classify_task],
        process=Process.sequential,
        verbose=False,
    )
    classify_out = await classify_crew.kickoff_async()
    cat_raw = str(classify_out.raw).strip().lower()
    if "bill" in cat_raw:
        category = "billing"
    elif "tech" in cat_raw:
        category = "technical"
    else:
        category = "general"
    specialist = specialists[category]
    print(f"  [ROUTED TO] → {specialist.role}")

    # Step 2: answer + format with the chosen specialist.
    # KB scoped to the category so the specialist has the same data the other
    # frameworks gave their nodes.
    answer_task = Task(
        description=(
            f"Customer question: {q}\n\n"
            f"Knowledge base for this category ({category}):\n{kbs[category]}\n\n"
            f"Answer using the knowledge base. Be specific and cite details from it."
        ),
        expected_output="A helpful answer to the customer's question",
        agent=specialist,
    )
    format_task = Task(
        description=(
            "Format this answer into a professional customer support response. "
            "Start with a greeting, include the answer, and end with an offer for further help. "
            "Do not invent details that were not in the original answer."
        ),
        expected_output="A polished, professional customer support response",
        agent=formatter_agent,
        context=[answer_task],
    )
    crew = Crew(
        agents=[specialist, formatter_agent],
        tasks=[answer_task, format_task],
        process=Process.sequential,
        verbose=False,
    )
    start = time.time()
    result = await crew.kickoff_async()
    elapsed = time.time() - start
    print(f"\nRESPONSE ({elapsed:.1f}s): {str(result)[:300]}")
    crewai_results.append(str(result)[:200])

CREWAI — CUSTOMER SUPPORT TRIAGE

>>> CUSTOMER: I can't log into my account, it keeps saying invalid password
  [ROUTED TO] → Technical Support Engineer

RESPONSE (69.6s): Hello,

I understand you are experiencing difficulties logging into your account with an 'invalid password' message. Here are some steps based on information available in our technical knowledge base that may help resolve the issue:

1. **Try resetting your password via:** Visit the URL `/reset`. Up

>>> CUSTOMER: I was charged twice for my last order #12345
  [ROUTED TO] → Billing Specialist

RESPONSE (45.5s): Hello! Thank you for bringing this issue to our attention. It seems that there was indeed a duplicate charge on your last order #12345. We investigate such issues within 24 hours and would appreciate if you could provide your order number so we can expedite the resolution process.

Once we have your

>>> CUSTOMER: What are your business hours?
  [ROUTED TO] → Customer Service Representative

RESPONSE (22.1s): 

In [14]:
# ── Honest line counts: measured, not estimated ──
# Counts non-blank, non-comment, non-import lines per framework cell.
# Note: OpenAI SDK cell hosts the shared KB + test_queries, so its count is
# inflated by ~10 lines that the other two frameworks reuse.
def _loc(src):
    n = 0
    for line in src.splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if s.startswith("import ") or s.startswith("from "):
            continue
        n += 1
    return n

# ponytail: self-check — 2 real lines, ignores comment/import/blank
assert _loc("a = 1\n# comment\nimport os\n\nb = 2") == 2, "loc miscounts"

def _find_cell(marker):
    for src in In:
        if marker in src:
            return src
    return ""

openai_loc    = _loc(_find_cell("OPENAI AGENTS SDK — CUSTOMER SUPPORT TRIAGE"))
langgraph_loc = _loc(_find_cell("LANGGRAPH — CUSTOMER SUPPORT TRIAGE"))
crewai_loc    = _loc(_find_cell("CREWAI — CUSTOMER SUPPORT TRIAGE"))

print(f"OpenAI SDK LOC:    {openai_loc}")
print(f"LangGraph LOC:     {langgraph_loc}")
print(f"CrewAI LOC:        {crewai_loc}")

OpenAI SDK LOC:    105
LangGraph LOC:     74
CrewAI LOC:        94


---
## The Honest Comparison

| Criterion | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| **Lines of code** | see measurement cell above | see measurement cell above | see measurement cell above |
| **Setup ceremony** | Low — Agent + handoffs | Medium — State + graph wiring | Medium — manual classify + 2 crews |
| **Routing logic** | LLM decides (implicit) | Explicit `conditional_edges` | Manual: classify crew → pick specialist in Python |
| **Control flow** | AI-driven, less predictable | Deterministic graph | Sequential task chains |
| **Adding new specialist** | Add Agent + handoff | Add node + edge + route | Add Agent + `specialists` dict entry |
| **State management** | Automatic (conversation) | Explicit `TypedDict` state | Via task `context` |
| **How KB reaches the model** | Via tool call (`lookup_*`) — model must choose to call it | Baked into the node's prompt — no second tool call needed | Baked into the task description — no second tool call needed |
| **Answer quality in this run** | Generic / escalation-style | Best — clean KB citation | Good — cites KB; formatter adds some flourish |
| **Debugging** | Runner traces | Graph visualization | Verbose logging |
| **Best for this problem** | Dynamic routing | Complex multi-step flows | Role-based pipelines |

### Lines of Code

See the measurement cell above for actual counts (non-blank, non-comment, non-import).
The OpenAI SDK cell hosts the shared `*_kb` dicts and `test_queries`, which inflates
its count by ~10 lines that the other two frameworks reuse — read the numbers with
that asymmetry in mind.

### Developer Experience Scores (1-5)

| Aspect | OpenAI SDK | LangGraph | CrewAI |
|---|---|---|---|
| Ease of setup | 4 | 3 | 3 |
| Readability | 4 | 3 | 4 |
| Flexibility | 4 | 5 | 3 |
| Debuggability | 3 | 5 | 4 |
| Production readiness | 4 | 5 | 3 |

**Honest take:** For a simple triage system, all three route correctly.
LangGraph wins on control, debuggability, and — in this run — answer quality.
The reason is mechanical: each LangGraph specialist node gets the KB baked
into its prompt, so the model answers in one shot. No second tool call required.
CrewAI matches that pattern once you pass the KB into the task description,
and the latest run shows it citing the KB correctly on all three queries —
though the formatter role still improvises headers ("Subject: ...") with a
small model. Routing isn't built in; you wire it yourself.
OpenAI SDK is the most AI-native and the most dependent on the model's
tool-calling reliability. Handoffs succeed, but the specialist then has to
*call its own* `lookup_*` tool — and with `qwen2.5:3b` that second call often
doesn't happen, so responses read as "we'll look into it" escalations instead
of KB-grounded answers. The cell prints `[WARN]` next to the routed agent
when it detects this. A larger model would close the gap.
The gap widens as complexity grows — that's what Days 21-30 explore.

## Key Takeaway

Building the same system 3 ways reveals that:
- **CrewAI** gives the clearest role separation but has no built-in router — you wire routing yourself, and you have to remember to pass the knowledge base into the task description
- **LangGraph** gives you the most control via explicit graph topology, and wins on answer quality in this run because each specialist node gets the KB in its prompt
- **OpenAI SDK** is the most AI-native — handoffs work, but the specialist still has to choose to call its lookup tool, and small models often don't

No framework is "best" — each optimizes for different developer priorities.
The right choice depends on your team, your problem complexity, and your
production requirements.

---
